# Step 1.2: Data Ingestion Testing

This notebook verifies the hybrid ingestion strategy:
1. **Yahoo Finance**: NIFTY50, Gold, USDINR
2. **Kite Connect**: India VIX

**Note:** Ensure you have updated `config/config.yaml` with your Kite credentials before running.

In [ ]:
import sys
import os

# Ensure the project root is in the path
sys.path.append(os.path.abspath('..'))

from src.data.data_manager import DataManager
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_theme(style="whitegrid")

## 1. Initialize Data Manager
This will load the config and initialize the ingestors.

In [ ]:
manager = DataManager(config_path="../config/config.yaml")

## 2. Fetch All Data
This will call the underlying Yahoo and Kite ingestors.

In [ ]:
try:
    df_raw = manager.fetch_all_data()
    print("Data fetched successfully!")
    display(df_raw.head())
    display(df_raw.tail())
except Exception as e:
    print(f"Ingestion failed: {e}")
    print("\nTIP: Check if your Kite Access Token is set correctly in config/config.yaml")

## 3. Data Integrity Check
Check for nulls and plot the series to ensure alignment.

In [ ]:
if 'df_raw' in locals() and not df_raw.empty:
    print("--- Missing Values ---")
    print(df_raw.isnull().sum())
    
    # Normalized plot for visual verification
    df_norm = (df_raw / df_raw.iloc[0]) * 100
    plt.figure(figsize=(15, 7))
    for col in df_norm.columns:
        plt.plot(df_norm.index, df_norm[col], label=col)
    
    plt.title("Asset Trinity + India VIX (Base 100)")
    plt.legend()
    plt.show()
else:
    print("No data available to plot.")

## 4. Persist to Disk
Save the cleaned raw data for Step 1.3.

In [ ]:
if 'df_raw' in locals() and not df_raw.empty:
    manager.save_raw_data(df_raw, "raw_trinity.csv")
    print("Raw data persisted to data/raw/raw_trinity.csv")